# HFR calibration — 0D model
Calibrate the undetermined ohmic-resistance parameters of the Pukrushpan-style 0D PEMFC model (`PEMFC_0D`) by fitting the simulated HFR — `sum(Rmem) + Re` at the final integration step — to the experimental HFR. Compare the calibration outcome across the same five experimental subsets used in the 1D notebooks.

In [ ]:
import sys
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import optuna
from copy import deepcopy
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# --------------- Set up project root path  --------------- #
project_folder_name = "MFC2024"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == project_folder_name), None)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import *
from model.dualscale import PEMFC_0D
from model.coefficients import *
from model.kinetic_eq import Rproton
from config.initialize import *

## Calibration setup

In [ ]:
# Tested currents and the (T, P, RHC) grid we sweep over.
I_tested   = [10, 20, 30, 40, 50]
RHC_tested = [0, 50]
PAC_tested = [1.3e5, 1.4e5, 1.5e5]
TFC_tested = [50, 60, 70]

# The HFR.xlsx file holds STACK-level HFR (22 cells in series). The
# ``n_cell`` constant in coefficients.py is 1 (number of cells in the
# SIMULATED stack), so the experimental HFR must be scaled with the actual
# experimental stack size to compare per-cell.
N_CELL_EXP = 22

### Experimental data import

In [ ]:
# Load the HFR experimental data. Each sheet name is "T{C}_P{mbar_gauge}_HRC{%}".
# The R column stores a "(R, I)" tuple as a string -- pull out the float R.
hfr_data_path = project_root / "data" / "HFR.xlsx"
hfr_testdata  = pd.read_excel(hfr_data_path, sheet_name=None)
hfrdata_exp   = {}
for sheet_name, raw in hfr_testdata.items():
    parsed = raw["R"].str.strip("()").str.split(",", expand=True).astype(float)
    hfrdata_exp[sheet_name] = pd.DataFrame({"I_LOAD": raw["I_LOAD"], "R": parsed[0]})
print(f"Loaded {len(hfrdata_exp)} HFR sheets: {list(hfrdata_exp.keys())}")

### Define `hfrtest_sim` and the objective factory

In [ ]:
def hfrtest_sim(params_trial, op_trial, cond_filter=None, lenient=False):
    """Run the 0D model across every (RHC, P, T) condition for which we have
    experimental HFR data and return ``{cond_key: [HFR at I_tested]}``
    in Ohm.m^2 per cell (HFR = Rmem + Re at the final integration step).

    Two failure modes:
    - ``lenient=False`` (default, used by the optimizer): if *any* single
      condition fails the whole sweep aborts and the function returns
      ``False``. The objective then penalises the trial.
    - ``lenient=True`` (used for the post-calibration ``sim_all`` plot): a
      failed condition is skipped and the function keeps going so that the
      successful conditions still show up in the fit plots.

    Pass ``cond_filter(T, P, RHC) -> bool`` to restrict which conditions are
    simulated.
    """
    Re = params_trial["Re"]
    result = {}
    for RHC in RHC_tested:
        for P_des in PAC_tested:
            for T_des in TFC_tested:
                cond_key = "T" + str(T_des) + "_P" + str(int(P_des/1e2 - 1e3)) + "_HRC" + str(RHC)
                if cond_key not in hfrdata_exp:
                    continue
                if cond_filter is not None and not cond_filter(T_des, P_des, RHC):
                    continue
                op = dict(op_trial)
                op["Phi_a_des"] = RHC / 100
                op["Phi_c_des"] = RHC / 100
                op["Pa_des"]    = P_des
                op["Pc_des"]    = P_des
                op["Tfc"]       = T_des + 273.15

                hfr_test = []
                cond_failed = False
                for I_LOAD in I_tested:
                    i_density = I_LOAD / params_trial["Aact"]
                    op["current_density"] = lambda t, _i=i_density: _i
                    try:
                        model = PEMFC_0D(params_trial, op)
                        y0 = model.default_initial_state(params_trial, op)
                        info = model.solve(t_span=(0, 60), y0=y0, method="BDF",
                                           max_step=0.1, verbose=False, sparsity=False,
                                           atol=1e-8, rtol=1e-6)
                        sol = info["sol"]
                        if sol is None or not sol.success:
                            cond_failed = True
                            break
                        last = {k: sol.y[idx, -1] for idx, k in enumerate(model.variable_names)}
                        # Map the 0D lumped lambda_mem onto the 1D-style keys
                        # that Rproton() expects.
                        last_for_R = {
                            "lambda_acl":   last["lambda_mem"],
                            "lambda_ccl":   last["lambda_mem"],
                            "lambda_mem_1": last["lambda_mem"],
                        }
                        Rmem_t, _, _ = Rproton(last_for_R, {**params_trial, "n_mem": 1}, op)
                        hfr_predicted = float(sum(Rmem_t)) + Re
                        if not math.isfinite(hfr_predicted):
                            cond_failed = True
                            break
                        hfr_test.append(hfr_predicted)
                    except Exception:
                        cond_failed = True
                        break

                if cond_failed:
                    if lenient:
                        continue   # skip this cond, keep sweeping
                    return False   # strict: abort the whole sweep
                result[cond_key] = hfr_test
    return result

In [ ]:
def _experimental_hfr(cond_key):
    """Experimental per-cell HFR at every I_tested for one condition.
    Convert from stack-level mOhm (22 cells in series) to per-cell
    Ohm.m^2:  R[mOhm] * 1e-3 / N_CELL_EXP * Aact.
    """
    df = hfrdata_exp[cond_key]
    i_exp = df["I_LOAD"].to_numpy(dtype=float)
    r_exp = df["R"].to_numpy(dtype=float) * 1e-3 / N_CELL_EXP * parameters["Aact"]
    idx = [np.argmin((i_test - i_exp) ** 2) for i_test in I_tested]
    return r_exp[idx]


def make_objective(cond_filter):
    """Return an optuna objective that fits only the conditions selected by
    ``cond_filter`` -- so a single objective function can be reused across
    the calibration scenarios."""
    def objective(trial):
        params_trial = deepcopy(parameters)
        params_trial["n_group_pt"] = 10
        op_trial     = deepcopy(operating_inputs)

        params_trial["Re"]          = trial.suggest_float("Re",          1e-7, 5e-6, log=True)
        params_trial["epsilon_gdl"] = trial.suggest_float("epsilon_gdl", 0.5,  0.8)
        params_trial["epsilon_mc"]  = trial.suggest_float("epsilon_mc",  0.15, 0.4)
        params_trial["epsilon_cl"]  = trial.suggest_float("epsilon_cl",  0.1,  0.4)
        params_trial["epsilon_c"]   = trial.suggest_float("epsilon_c",   0.2,  0.3)
        params_trial["tau"]         = trial.suggest_float("tau",         1.0,  4.0)
        params_trial["Hgdl"]        = trial.suggest_float("Hgdl",        2e-5, 5e-5, log=True)
        params_trial["Hcl"]         = trial.suggest_float("Hcl",         1e-5, 2e-5, log=True)

        try:
            sim = hfrtest_sim(params_trial, op_trial, cond_filter=cond_filter)
        except Exception:
            return 1e6
        if sim is False or not sim:
            return 1e6

        error = 0.0
        for cond_key, r_sim in sim.items():
            r_exp = _experimental_hfr(cond_key)
            error += float(np.sum((np.array(r_sim) - r_exp) ** 2))
        return error
    return objective

## Calibration scenarios

In [ ]:
SINGLE_T   = 60       # degrees C
SINGLE_P   = 1.4e5    # Pa
SINGLE_RHC = 50       # %
TWO_HUM_T  = 50       # degrees C
TWO_HUM_P  = 1.3e5    # Pa
FIXED_P    = 1.3e5    # Pa
FIXED_T    = 60       # degrees C

scenarios = {
    "one_condition":      {
        "label":  f"One condition (T={SINGLE_T}, P={SINGLE_P/1e5:.1f} bar, RHC={SINGLE_RHC})",
        "filter": lambda T, P, RHC: (T == SINGLE_T) and (P == SINGLE_P) and (RHC == SINGLE_RHC),
    },
    "two_humidification": {
        "label":  f"Two humidifications (P={TWO_HUM_P/1e5:.1f} bar, T={TWO_HUM_T}, RHC in {{0, 50}})",
        "filter": lambda T, P, RHC: (T == TWO_HUM_T) and (P == TWO_HUM_P) and (RHC in (0, 50)),
    },
    "fixed_pressure":     {
        "label":  f"Fixed pressure (P={FIXED_P/1e5:.1f} bar, all T, all RHC)",
        "filter": lambda T, P, RHC: P == FIXED_P,
    },
    "fixed_temperature":  {
        "label":  f"Fixed temperature (T={FIXED_T}, all P, all RHC)",
        "filter": lambda T, P, RHC: T == FIXED_T,
    },
    "all_data":           {
        "label":  "All conditions",
        "filter": lambda T, P, RHC: True,
    },
}


def conditions_in_scenario(filter_fn):
    keys = []
    for RHC in RHC_tested:
        for P_des in PAC_tested:
            for T_des in TFC_tested:
                cond_key = "T" + str(T_des) + "_P" + str(int(P_des/1e2 - 1e3)) + "_HRC" + str(RHC)
                exp_data = globals().get("polardata_exp") or globals().get("hfrdata_exp") or {}
                if cond_key in exp_data and filter_fn(T_des, P_des, RHC):
                    keys.append(cond_key)
    return keys


for name, info in scenarios.items():
    matches = conditions_in_scenario(info["filter"])
    print(f"  {name:20s}  {len(matches):2d} conditions  -> {matches}")

In [ ]:
# Run a separate optuna study per scenario.
optuna.logging.set_verbosity(optuna.logging.WARNING)

PER_SCENARIO_TIMEOUT = 300     # seconds
PER_SCENARIO_TRIALS  = 2000
N_JOBS               = 6

results = {}
for name, info in scenarios.items():
    print(f"\n=== Calibrating scenario: {name} ===")
    print(f"    {info['label']}")
    print(f"    conditions used: {conditions_in_scenario(info['filter'])}")
    study = optuna.create_study(direction="minimize")
    study.optimize(
        make_objective(info["filter"]),
        n_trials=PER_SCENARIO_TRIALS,
        timeout=PER_SCENARIO_TIMEOUT,
        n_jobs=N_JOBS,
        show_progress_bar=False,
    )

    # Re-run hfrtest_sim with the best params over ALL conditions, in lenient
    # mode -- a single bad condition no longer wipes the rest off the plot.
    params_best = deepcopy(parameters)
    params_best["n_group_pt"] = 10
    op_best     = deepcopy(operating_inputs)
    params_best.update(study.best_params)
    sim_all     = hfrtest_sim(params_best, op_best, cond_filter=None, lenient=True)

    results[name] = {
        "label":           info["label"],
        "calibrated_keys": conditions_in_scenario(info["filter"]),
        "best_params":     study.best_params,
        "best_value":      study.best_value,
        "sim_all":         sim_all,
    }
    print(f"    best objective on the calibrated subset: {study.best_value:.3e}")
    n_ok = len(sim_all); n_total = len(hfrdata_exp)
    print(f"    sim_all covered {n_ok}/{n_total} conditions")

## Best parameters side-by-side

In [ ]:
param_df = pd.DataFrame({name: info["best_params"] for name, info in results.items()})
print(param_df.to_string(float_format=lambda v: f"{v:.4g}"))

## Compare best-fit HFR curves across scenarios
`plot_scenario_fit(name)` produces two figures per scenario: the fit on the conditions used during calibration, then the fit on every experimental condition.

In [ ]:
def plot_scenario_fit(name):
    """For one calibration scenario, show two figures: the HFR fit on the
    conditions used in the optimisation, then on every condition.
    Blue solid markers = experiment, red dotted squares = simulation.
    """
    info = results[name]
    cal_set = set(info["calibrated_keys"])

    panels_to_show = [
        ("USED for calibration", info["calibrated_keys"]),
        ("ALL conditions",       list(hfrdata_exp.keys())),
    ]
    for title_suffix, keys in panels_to_show:
        n = len(keys)
        if n == 0:
            print(f"({name}: no conditions for '{title_suffix}')")
            continue
        ncols = min(4, max(1, n))
        nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.4 * nrows),
                                 sharex=True, sharey=True, squeeze=False)
        for ax, cond_key in zip(axes.flatten(), keys):
            r_exp = _experimental_hfr(cond_key)
            r_sim = info["sim_all"].get(cond_key)
            ax.plot(I_tested, r_exp, "o-", color="tab:blue",
                    linewidth=1.4, label="experiment")
            if r_sim is not None:
                ax.plot(I_tested, r_sim, "s:", color="tab:red",
                        linewidth=1.2, label="simulated")
            in_cal = cond_key in cal_set
            tag = "[CAL] " if in_cal else ""
            ax.set_title(f"{tag}{cond_key}", fontsize=8,
                         color="tab:green" if in_cal else "black")
            ax.grid(True, alpha=0.3)
        for ax in axes.flatten()[n:]:
            ax.set_visible(False)
        axes.flatten()[0].legend(fontsize=8, loc="best")
        fig.suptitle(f"{info['label']}  --  {title_suffix}\n"
                     f"objective on calibrated subset = {info['best_value']:.3e}",
                     fontsize=10)
        fig.tight_layout()
        plt.show()

### Scenario: `one_condition`

In [ ]:
plot_scenario_fit("one_condition")

### Scenario: `two_humidification`

In [ ]:
plot_scenario_fit("two_humidification")

### Scenario: `fixed_pressure`

In [ ]:
plot_scenario_fit("fixed_pressure")

### Scenario: `fixed_temperature`

In [ ]:
plot_scenario_fit("fixed_temperature")

### Scenario: `all_data`

In [ ]:
plot_scenario_fit("all_data")

## Residual heatmap

In [ ]:
all_cond_keys = list(hfrdata_exp.keys())
n_scen = len(results)
residual_matrix = np.full((n_scen, len(all_cond_keys)), np.nan)
for i_scen, (name, info) in enumerate(results.items()):
    for i_cond, cond_key in enumerate(all_cond_keys):
        if cond_key not in info["sim_all"]:
            continue
        r_sim = np.array(info["sim_all"][cond_key])
        r_exp = _experimental_hfr(cond_key)
        residual_matrix[i_scen, i_cond] = float(np.sum((r_sim - r_exp) ** 2))

fig, ax = plt.subplots(figsize=(max(8, len(all_cond_keys) * 0.5), 0.5 + 0.5 * n_scen))
im = ax.imshow(residual_matrix, aspect="auto", cmap="cividis_r")
ax.set_yticks(range(n_scen)); ax.set_yticklabels(list(results.keys()), fontsize=9)
ax.set_xticks(range(len(all_cond_keys)))
ax.set_xticklabels(all_cond_keys, rotation=45, ha="right", fontsize=8)
for i_scen, (name, info) in enumerate(results.items()):
    cal_set = set(info["calibrated_keys"])
    for i_cond, cond_key in enumerate(all_cond_keys):
        if cond_key in cal_set:
            ax.add_patch(plt.Rectangle((i_cond - 0.5, i_scen - 0.5), 1, 1,
                                       fill=False, edgecolor="white", linewidth=2))
ax.set_title("Sum-of-squared-errors per condition (white box = used during calibration)")
fig.colorbar(im, ax=ax, label=r"$\Sigma(\Delta R)^2$")
plt.tight_layout(); plt.show()